In [1]:
import os
import sys

import yaml
from ultralytics import RTDETR
import pandas as pd

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

In [25]:
train_df = pd.read_csv(config.TRAIN_CSV_PATH)
val_df = pd.read_csv(config.VALIDATION_CSV_PATH)


def generate_training_paths(df: pd.DataFrame) -> pd.DataFrame:
    """Generate full paths for training images."""
    # /data/horse/ws/kein254g-team_project/train/tomo_0a8f05/slice_0000.jpg
    # /data/horse/ws/kein254g-team_project/train/tomo_0a8f05/slice_0117.jpg
    # slice must be slice_XXXX with leading zeros

    slice_num = df["Motor axis 0"].apply(lambda x: f"slice_{int(x):04d}.jpg" if x != -1 else "")
    df["slice"] = slice_num

    df["image_path"] = df.apply(
        lambda row: os.path.join(config.TRAIN_DATASET_DIR, row["tomo_id"], row["slice"]), axis=1
    )

    df[["row_id", "Motor axis 0", "slice", "image_path"]].head()
    print(df["image_path"].head().iloc[1])
    return df


# Generate image paths for training and validation sets
generate_training_paths(train_df)
# print(generate_training_paths(val_df)['image_path'].head())

/data/horse/ws/kein254g-team_project/data/train/tomo_38d285/


,row_id,tomo_id,Motor axis 0,Motor axis 1,Motor axis 2,Array shape (axis 0),Array shape (axis 1),Array shape (axis 2),Voxel spacing,Number of motors,slice,image_path
0,672,tomo_e5ac94,161.0,481.0,342.0,300,960,928,13.1,1,slice_0161.jpg,/data/horse/ws/kein254g-team_project/data/trai...
1,171,tomo_38d285,-1.0,-1.0,-1.0,800,928,960,13.1,0,,/data/horse/ws/kein254g-team_project/data/trai...
2,316,tomo_6acb9e,80.0,562.0,831.0,300,960,928,13.1,1,slice_0080.jpg,/data/horse/ws/kein254g-team_project/data/trai...
3,449,tomo_9ed470,160.0,270.0,665.0,300,959,928,15.6,1,slice_0160.jpg,/data/horse/ws/kein254g-team_project/data/trai...
4,650,tomo_dee783,-1.0,-1.0,-1.0,300,960,928,15.6,0,,/data/horse/ws/kein254g-team_project/data/trai...
...,...,...,...,...,...,...,...,...,...,...,...,...
510,396,tomo_8d5995,-1.0,-1.0,-1.0,300,960,928,15.6,0,,/data/horse/ws/kein254g-team_project/data/trai...
511,129,tomo_2bb588,97.0,709.0,611.0,300,960,928,13.1,1,slice_0097.jpg,/data/horse/ws/kein254g-team_project/data/trai...
512,545,tomo_bbe766,195.0,855.0,288.0,300,959,928,15.6,1,slice_0195.jpg,/data/horse/ws/kein254g-team_project/data/trai...
513,87,tomo_1f0e78,112.0,797.0,264.0,300,959,928,15.6,1,slice_0112.jpg,/data/horse/ws/kein254g-team_project/data/trai...


In [2]:
data_cfg = {
    "path": config.DATASET_DIR,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 1,  # number of classes
    "names": ["motor"],  # class names
}

with open(f"{config.SRC}/data.yaml", "w") as f:
    yaml.dump(data_cfg, f)

print("data.yaml written successfully")

data.yaml written successfully


In [3]:
model = RTDETR("rtdetr-l.pt")
results = model.train(
    data=f"{config.SRC}/data.yaml",
    project=config.RTDETR_RESULT,
    name="motor_rtdetr_l_1024",
    epochs=2,
    patience=30,
    batch=16,
    imgsz=1024,
    device="cuda",
    optimizer="AdamW",
    lr0=1e-4,
    lrf=0.1,
    cos_lr=True,
    warmup_epochs=3,
    cache="disk",
    augment=True,
    hsv_h=0.0,
    hsv_s=0.1,
    hsv_v=0.2,
    degrees=0.0,
    translate=0.05,
    scale=0.10,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.5,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    rect=True,
    multi_scale=False,
    save_period=1,
    verbose=True,
)
print(results)

New https://pypi.org/project/ultralytics/8.3.203 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.196 🚀 Python-3.9.21 torch-2.8.0+cu128 


/home/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/.venv/lib64/python3.9/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.
